## Apex Wealth Data Pipeline

In [1]:
# Import necessary libraries
import requests
import pandas as pd
import psycopg2 as pg
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os


In [2]:
# Load environment variables from .env file
load_dotenv()

# Get API key from environment variable
api_key = os.getenv('API_KEY')

In [3]:
api_key

'ed9a538367064117bf0e2af332ee25d4'

## Extraction

In [4]:
# define url endpoint
symbol = 'AAPL'
url = f'https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}'

In [5]:
# Making the API request from the server
response = requests.get(url)

# Check if the request was successful
response.status_code

200

In [6]:
 # Get the JSON data from the response
data = response.json()
data

{'meta': {'symbol': 'AAPL',
  'interval': '1min',
  'currency': 'USD',
  'exchange_timezone': 'America/New_York',
  'exchange': 'NASDAQ',
  'mic_code': 'XNGS',
  'type': 'Common Stock'},
 'values': [{'datetime': '2026-04-14 15:59:00',
   'open': '258.75500',
   'high': '258.85999',
   'low': '258.69000',
   'close': '258.85501',
   'volume': '828866'},
  {'datetime': '2026-04-14 15:58:00',
   'open': '258.62000',
   'high': '258.76001',
   'low': '258.58',
   'close': '258.75500',
   'volume': '325408'},
  {'datetime': '2026-04-14 15:57:00',
   'open': '258.69000',
   'high': '258.715',
   'low': '258.62000',
   'close': '258.62000',
   'volume': '249166'},
  {'datetime': '2026-04-14 15:56:00',
   'open': '258.59000',
   'high': '258.69000',
   'low': '258.54999',
   'close': '258.68500',
   'volume': '199315'},
  {'datetime': '2026-04-14 15:55:00',
   'open': '258.22000',
   'high': '258.63000',
   'low': '258.22000',
   'close': '258.57999',
   'volume': '265875'},
  {'datetime': '20

In [7]:
# Calling the 'values' key from the JSON data to get the time series data

data['values']

[{'datetime': '2026-04-14 15:59:00',
  'open': '258.75500',
  'high': '258.85999',
  'low': '258.69000',
  'close': '258.85501',
  'volume': '828866'},
 {'datetime': '2026-04-14 15:58:00',
  'open': '258.62000',
  'high': '258.76001',
  'low': '258.58',
  'close': '258.75500',
  'volume': '325408'},
 {'datetime': '2026-04-14 15:57:00',
  'open': '258.69000',
  'high': '258.715',
  'low': '258.62000',
  'close': '258.62000',
  'volume': '249166'},
 {'datetime': '2026-04-14 15:56:00',
  'open': '258.59000',
  'high': '258.69000',
  'low': '258.54999',
  'close': '258.68500',
  'volume': '199315'},
 {'datetime': '2026-04-14 15:55:00',
  'open': '258.22000',
  'high': '258.63000',
  'low': '258.22000',
  'close': '258.57999',
  'volume': '265875'},
 {'datetime': '2026-04-14 15:54:00',
  'open': '258.45499',
  'high': '258.60999',
  'low': '258.20499',
  'close': '258.20499',
  'volume': '170274'},
 {'datetime': '2026-04-14 15:53:00',
  'open': '258.31500',
  'high': '258.46',
  'low': '258

In [8]:
# get the 'high' price from the second entry in the 'values' list

data['values'][1]['high']

'258.76001'

## Transformation

In [9]:
def transform_data(data):

    # extract values from json
    time_series = data['values']

    # convert to dataframe
    df = pd.DataFrame(time_series)

    # convert columns to correct data types
    df['datetime'] = pd.to_datetime(df['datetime'])

    # convert everything else
    df = df.astype({
        'open': 'float',
        'high': 'float',
        'low': 'float',
        'close': 'float',
        'volume': 'int'
    })
    return df

In [10]:
# calling the transform function
df = transform_data(data)
df

,datetime,open,high,low,close,volume
0,2026-04-14 15:59:00,258.75500,258.85999,258.69000,258.85501,828866
1,2026-04-14 15:58:00,258.62000,258.76001,258.58000,258.75500,325408
2,2026-04-14 15:57:00,258.69000,258.71500,258.62000,258.62000,249166
3,2026-04-14 15:56:00,258.59000,258.69000,258.54999,258.68500,199315
4,2026-04-14 15:55:00,258.22000,258.63000,258.22000,258.57999,265875
5,2026-04-14 15:54:00,258.45499,258.60999,258.20499,258.20499,170274
6,2026-04-14 15:53:00,258.31500,258.46000,258.31009,258.45999,165683
7,2026-04-14 15:52:00,258.37000,258.42001,258.30371,258.31500,84886
8,2026-04-14 15:51:00,258.50000,258.51000,258.22000,258.35999,119398
9,2026-04-14 15:50:00,258.26999,258.70000,258.26000,258.50000,183531


## Loading

In [11]:
load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

In [12]:
DB_NAME

'apex_db_eu'

In [13]:
# create connection url
db_url = f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

# create sqlalchemy engine
engine = create_engine(db_url)

# load dataframe to sql table
df.to_sql('stock_prices', engine, if_exists='append', index=False)

print("Data has been successfully loaded to the database.")

Data has been successfully loaded to the database.


In [14]:
# save as csv file
df.to_csv('stock_prices.csv', index=False)
print("Data has been successfully saved to CSV file.")  

Data has been successfully saved to CSV file.
